# Operations Research Group Project

## Scope

**Objective Function**: 

$$\min_{x} \quad Z = \sum_{i \in I} c_i x_i$$

**Index Sets and Variables**

$i \in I = \{1, 2, \ldots, 204\}$ — food items

$j \in J = \{1, 2, \ldots, 29\}$ — nutrients

$g \in G$ — food groups for meal realism constraints

$x_i \geq 0 \quad \forall i \in I$

**Parameters**:

$c_i$ — cost per gram of food $i$ (USD/g)

$a_{ij}$ — amount of nutrient $j$ per gram of food $i$ (nutrient unit/g)

$b_j^{LB}$ — lower bound for nutrient $j$ per meal

$b_j^{UB}$ — upper bound for nutrient $j$ per meal

$U_g$ — combined gram cap for food group $g$

$I_g \subseteq I$ — set of food indices belonging to group $g$

**Constraints**:

$$\sum_{i \in I} a_{ij} x_i \geq b_j^{LB} \quad \forall j \in J_{\min}$$

$$\sum_{i \in I} a_{ij} x_i \leq b_j^{UB} \quad \forall j \in J_{\max}$$

$$b_j^{LB} \leq \sum_{i \in I} a_{ij} x_i \leq b_j^{UB} \quad \forall j \in J_{\text{range}}$$

$$\sum_{i \in I_g} x_i \leq U_g \quad \forall g \in G$$

$$0 \leq x_i \leq 200 \quad \forall i \in I$$

**Food Group Caps** $U_g$:

| Group $g$ | $\lvert I_g \rvert$ | $U_g$ (g) | Rationale |
|---|---|---|---|
| Fluid milk | all milk variants | 150 | One school carton |
| Cooking oils | all pure oils | 3 | Incidental cooking use |
| Grain products | all grains and cereals | 120 | One grain serving |
| Leafy vegetables | all leafy greens | 80 | One side salad |
| Protein foods | all meat, fish, eggs | 100 | One protein serving |
| Fruits | all fresh fruits | 120 | One fruit serving |

In [1]:
# Necessary imports
import numpy as np
import pandas as pd

from scipy.optimize import linprog, OptimizeResult
from tabulate import tabulate

import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Data loc
PATH_PRICES = "food_prices_200.xlsx"
PATH_NUTRITION = "food_nutrition_200.xlsx"
PATH_REQ_LB = "dri_lunch_lb.csv"
PATH_REQ_UB = "dri_lunch_ub.csv"

# Portion cap to prevent any giant meal
MAX_GRAMS_PER_FOOD = 200.0

In [3]:
# Data load
def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Loading all required data

    Tap water should be excluded

    Returns:
    nutrition   : (204, 35) - food nutritional content per 100g
    prices      : (204, 13) - food prices per gram (Base on the US)
    lb          : (5, 29+4) - lower bounds per age-sex group
    ub          : (5, 29+4) - upper bounds per age-sex group
    """
    nutrition = pd.read_excel(PATH_NUTRITION, keep_default_na=False)
    prices = pd.read_excel(PATH_PRICES, keep_default_na=False)
    lb = pd.read_csv(PATH_REQ_LB).set_index("age_sex_group")
    ub = pd.read_csv(PATH_REQ_UB).set_index("age_sex_group")

    # Remove water
    nutrition = nutrition[nutrition["food_name"] != "Water, Tap"].reset_index(drop=True)
    prices = prices[prices["food_name"] != "Water, Tap"].reset_index(drop=True)

    return nutrition, prices, lb, ub

In [4]:
nutrition, prices, lb, ub = load_data()
print(f"Foods: {len(nutrition)}")
print(f"Nutrients: {nutrition.columns}")
print(f"Groups: {list(lb.index)}")

Foods: 204
Nutrients: Index(['food_id', 'food_name', 'category', 'subcategory', 'fdc_id',
       'preparation_state', 'energy_kcal', 'protein_g', 'total_fat_g',
       'carbohydrates_g', 'dietary_fiber_g', 'sugars_g', 'saturated_fat_g',
       'monounsaturated_fat_g', 'polyunsaturated_fat_g', 'cholesterol_mg',
       'calcium_mg', 'iron_mg', 'magnesium_mg', 'phosphorus_mg',
       'potassium_mg', 'sodium_mg', 'zinc_mg', 'vitamin_a_ug_rae',
       'vitamin_d_ug', 'vitamin_e_mg_ate', 'vitamin_k_ug', 'vitamin_c_mg',
       'thiamin_b1_mg', 'riboflavin_b2_mg', 'niacin_b3_mg', 'vitamin_b6_mg',
       'folate_ug_dfe', 'vitamin_b12_ug', 'selenium_ug'],
      dtype='object')
Groups: ['children_4_8_MF', 'males_9_13', 'females_9_13', 'males_14_18', 'females_14_18']


In [5]:
# Food group cap
def get_group_caps(food_names: list[str]) -> dict[str, tuple[list[int], float]]:
    """
    Define food group caps for meal realism.

    For each group: a list of food indices and a combined gram cap.
    The LP must spread across food groups — no single category can dominate.

    Groups and caps:
    fluid_milk      — one standard school carton (all milk types combined)
    cooking_oils    — incidental cooking use only
    grain_products  — one grain serving (all grains/cereals combined)
    leafy_veg       — one side salad portion
    protein_foods   — one protein serving
    fruits          — one fruit serving

    Returns:
    dict mapping group_name -> (list of food indices in group, gram cap)
    """
    # Keywords to identify
    group_keywords = {
        "fluid_milk"    : ["milk, 2%", "milk, whole", "milk, 1%", "milk, nonfat",
                           "milk, skim", "milk, soy", "milk, almond", "milk, chocolate",
                           "milk,"],
        "cooking_oils"  : ["vegetable oil", "olive oil", "canola oil", "coconut oil"],
        "grain_products": ["cornmeal", "corn flakes", "raisin bran", "oatmeal",
                           "granola", "rice, white", "rice, brown", "pasta",
                           "spaghetti", "macaroni", "bread,", "tortilla",
                           "bagel", "pita", "quinoa", "barley", "millet",
                           "couscous", "buckwheat", "crackers", "wild rice"],
        "leafy_veg"     : ["spinach", "kale", "romaine", "lettuce", "collard",
                           "swiss chard", "bok choy", "mixed greens"],
        "protein_foods" : ["chicken", "beef", "pork", "turkey", "tuna",
                           "salmon", "tilapia", "cod", "shrimp", "sardine",
                           "mackerel", "clam", "crab", "catfish", "liver",
                           "egg", "tofu", "edamame"],
        "fruits"        : ["banana", "apple", "orange", "grape", "strawberr",
                           "blueberr", "peach", "pear", "pineapple", "watermelon",
                           "cantaloupe", "mango", "kiwi", "cherry", "plum",
                           "grapefruit", "pomegranate", "raspberry"],
    }
    
    group_caps = {
        "fluid_milk"    : 150.0,
        "cooking_oils"  : 3.0,
        "grain_products": 120.0,
        "leafy_veg"     : 80.0,
        "protein_foods" : 100.0,
        "fruits"        : 120.0,
    }

    groups = {}
    food_names_lower = [f.lower() for f in food_names]

    for group, keywords in group_keywords.items():
        indices = [
            i for i, name in enumerate(food_names_lower)
            if any(kw in name for kw in keywords)
        ]
        groups[group] = (indices, group_caps[group])

    return groups

In [6]:
# Build LP coefficient matrices
def build_matrices(
        nutrition: pd.DataFrame,
        prices: pd.DataFrame
) -> tuple[np.ndarray, np.ndarray, list[str], list[str]]:
    """
    Constructing the obj vec c and const coeff matrix A

    Notation:
    n   : number of foods
    m   : number of nutrients
    x   : decision variable vector (grams of each food)
    c   : cost per gram (USD/g)
    A   : nutrient per gram

    Returns:
    c           : (n,) - obj coeff (USD per gram)
    A           : (n, m) - nutrient per gram matrix
    food_names  : list[str]
    nut_cols    : list[str]
    """
    nut_cols = list(nutrition.columns[6:]) # 29 nutrients

    # Merge on food_id
    merged = nutrition.merge(
        prices[["food_id", "price_per_1g_usd"]],
        on="food_id",
        how="inner"
    )

    food_names = list(merged["food_name"])
    c = merged["price_per_1g_usd"].to_numpy(dtype=float)
    A = merged[nut_cols].to_numpy(dtype=float) / 100.0  # to be 1g

    return c, A, food_names, nut_cols

In [7]:
c, A, food_names, nut_cols = build_matrices(nutrition, prices)
print(f"Obj vector c: {c.shape}     (USD/g)")
print(f"const mat A: {A.shape}      (nutrient/g)")
print(f"Cost range: ${c.min():.4f} - ${c.max():.4f} per gram")

Obj vector c: (204,)     (USD/g)
const mat A: (204, 29)      (nutrient/g)
Cost range: $0.0009 - $0.0500 per gram


In [8]:
groups = get_group_caps(food_names)

for group, (indices, cap) in groups.items():
    print(f"\n{group} (cap: {cap} g) — {len(indices)} foods:")
    for i in indices:
        print(f"  [{i}] {food_names[i]}")


fluid_milk (cap: 150.0 g) — 7 foods:
  [65] Milk, Whole (3.25% fat), Fluid
  [66] Milk, 2% Reduced Fat, Fluid
  [67] Milk, 1% Low-Fat, Fluid
  [68] Milk, Nonfat/Skim, Fluid
  [69] Milk, Soy, Unsweetened (fortified)
  [70] Milk, Almond, Unsweetened (fortified)
  [162] Milk, Chocolate, Whole (1% fat)

cooking_oils (cap: 3.0 g) — 4 foods:
  [150] Olive Oil, Extra Virgin
  [151] Canola Oil
  [152] Vegetable Oil (Soybean/Canola blend)
  [153] Coconut Oil (refined)

grain_products (cap: 120.0 g) — 27 foods:
  [60] Pumpkin Seeds (pepitas, roasted)
  [80] Bread, White, Pan (commercially prepared)
  [81] Bread, Whole Wheat (commercially prepared)
  [82] Bagel, Plain (commercially prepared)
  [84] Pita Bread, White
  [85] Tortilla, Flour (10 ct, 8-inch)
  [86] Tortilla, Corn (6-inch, soft)
  [87] Rice, White, Long Grain (cooked)
  [88] Rice, Brown, Long Grain (cooked)
  [89] Pasta, Spaghetti, Dry, Enriched
  [90] Pasta, Whole Wheat, Dry
  [91] Macaroni & Cheese (from box, prepared)
  [92] Oatme

In [9]:
# Build constraints
def get_constraints() -> dict[str, list[str]]:
    """
    Classifying each nutrients

    Categories:
    min_only    : sum_i a_ij x_i >= LB_j   (RDA/AI minimum requirements)
    max_only    : sum_i a_ij x_i <= UB_j   (NSLP/DGA ceilings)
    range       : LB_j <= sum_i a_ij x_i <= UB_j  (energy + fat)
    skip        : no numeric DRI -- excluded from LP constraints
    """
    min_only = [
        "protein_g",
        "carbohydrates_g",
        "dietary_fiber_g",
        "calcium_mg",
        "iron_mg",
        "magnesium_mg",
        "phosphorus_mg",
        "potassium_mg",
        "zinc_mg",
        "vitamin_a_ug_rae",
        "vitamin_d_ug",
        "vitamin_e_mg_ate",
        "vitamin_k_ug",
        "vitamin_c_mg",
        "thiamin_b1_mg",
        "riboflavin_b2_mg",
        "niacin_b3_mg",
        "vitamin_b6_mg",
        "folate_ug_dfe",
        "vitamin_b12_ug",
        "selenium_ug",
    ]
    max_only = [
        "sugars_g",         # added sugars  < 10% of meal kcal (DGA 2020-2025)
        "saturated_fat_g",  # saturated fat < 10% of meal kcal (DGA 2020-2025)
        "sodium_mg",        # NSLP grade-specific sodium ceiling
    ]

    range_nuts = [
        "energy_kcal",      # NSLP calorie window (e.g. 750-850 kcal for grades 9-12)
        "total_fat_g",      # AMDR 25-35% of meal energy
    ]
    skip = [
        "monounsaturated_fat_g",   # no numeric DRI established
        "polyunsaturated_fat_g",   # no numeric DRI established
        "cholesterol_mg",          # DGA: minimise, no numeric constraint
    ]
 
    return {
        "min_only"  : min_only,
        "max_only"  : max_only,
        "range"     : range_nuts,
        "skip"      : skip,
    }

In [10]:
constraint_sets = get_constraints()
for ctype, nuts in constraint_sets.items():
    print(f"{ctype}: {len(nuts):2d} nutrients")

min_only: 21 nutrients
max_only:  3 nutrients
range:  2 nutrients
skip:  3 nutrients


In [11]:
# Build linprog and solve
def solve_lp(
    group_id: str,
    c: np.ndarray,
    A: np.ndarray,
    food_names: list[str],
    nut_cols: list[str],
    lb_row: pd.Series,
    ub_row: pd.Series,
    constraint_sets: dict[str, list[str]],
    groups: dict,
) -> dict:
    """
    Build and solve the LP for one age-sex group with linprog

    Parameters:
    group_id        : age_sex_group label (e.g. 'males_14_18')
    c               : (n,) cost per gram
    A               : (n, m) nutrient per gram matrix
    food_names      : list of food labels (length n)
    nut_cols        : list of 29 nutrient column names
    lb_row          : Series of lower bounds for this group
    ub_row          : Series of upper bounds for this group
    constraint_sets : categorised nutrient lists from get_constraint_sets()
 
    Returns:
    dict with keys: group_id, status, cost_usd, foods (DataFrame),
                    nutrients (DataFrame)
    """
    n = len(food_names)
    nut_idx = {nut: j for j, nut in enumerate(nut_cols)}
 
    A_ub_rows = []
    b_ub_rows = []
 
    def _add_constraint(row_vec: np.ndarray, rhs: float) -> None:
        A_ub_rows.append(row_vec)
        b_ub_rows.append(rhs)

    def _valid(val) -> bool:
        """Return True if val is a usable numeric bound."""
        if val == "":
            return False
        try:
            return not np.isnan(float(val))
        except (TypeError, ValueError):
            return False
        
    # Min only constraints
    for nut in constraint_sets["min_only"]:
        j = nut_idx[nut]
        val = lb_row.get(nut, "")
        if not _valid(val):
            continue
        _add_constraint(-A[:, j], -float(val))

    # Max only constraints
    for nut in constraint_sets["max_only"]:
        j = nut_idx[nut]
        val = ub_row.get(nut, "")
        if not _valid(val):
            continue
        _add_constraint(A[:, j], float(val))

    # Range in both directions
    for nut in constraint_sets["range"]:
        j = nut_idx[nut]
        v_lb = lb_row.get(nut, "")
        v_ub = ub_row.get(nut, "")
        if _valid(v_lb):
            _add_constraint(-A[:, j], -float(v_lb))
        if _valid(v_ub):
            _add_constraint( A[:, j],  float(v_ub))
    
    # Group cap constraints: sum of x_i for all i in group G <= cap_G
    for group_name, (group_indices, cap) in groups.items():
        if not group_indices:
            continue
        row = np.zeros(n)
        for i in group_indices:
            row[i] = 1.0
        _add_constraint(row, cap)

    # Assemble and solve
    A_ub = np.array(A_ub_rows)
    b_ub = np.array(b_ub_rows)
    bounds = [(0.0, MAX_GRAMS_PER_FOOD)] * n
 
    res: OptimizeResult = linprog(
        c = c,
        A_ub = A_ub,
        b_ub = b_ub,
        bounds = bounds,
        method = "highs",
    )

    # Infeasibility
    if res.status != 0:
        return {
            "group_id" : group_id,
            "status" : res.message,
            "cost_usd" : None,
            "foods" : pd.DataFrame(),
            "nutrients": pd.DataFrame(),
        }
 
    portions = res.x
    cost = float(res.fun)

    # Selection of foods (portions > 0.1g threshold)
    mask = portions > 0.1
    food_df = pd.DataFrame({
        "food_name" : [food_names[i] for i in range(n) if mask[i]],
        "grams" : portions[mask].round(2),
        "cost_usd" : (portions[mask] * c[mask]).round(6),
        "cost_pct" : (portions[mask] * c[mask] / cost * 100).round(2),
    }).sort_values("cost_usd", ascending=False).reset_index(drop=True)

    # Nutrient totals vs bounds + binding
    achieved = A.T @ portions   # shape (m,)
    nut_rows = []
    for j, nut in enumerate(nut_cols):
        v_lb = lb_row.get(nut, "")
        v_ub = ub_row.get(nut, "")
        b_lb_f = float(v_lb) if _valid(v_lb) else np.nan
        b_ub_f = float(v_ub) if _valid(v_ub) else np.nan
 
        lb_slack   = round(achieved[j] - b_lb_f, 6) if not np.isnan(b_lb_f) else np.nan
        ub_slack   = round(b_ub_f - achieved[j], 6) if not np.isnan(b_ub_f) else np.nan
        lb_binding = (not np.isnan(lb_slack)) and (abs(lb_slack) < 0.005 * (abs(b_lb_f) + 1e-9))
        ub_binding = (not np.isnan(ub_slack)) and (abs(ub_slack) < 0.005 * (abs(b_ub_f) + 1e-9))
 
        nut_rows.append({
            "nutrient" : nut,
            "achieved" : round(achieved[j], 4),
            "lb" : b_lb_f,
            "ub" : b_ub_f,
            "lb_slack" : lb_slack,
            "ub_slack" : ub_slack,
            "lb_binding" : lb_binding,
            "ub_binding" : ub_binding,
        })
 
    return {
        "group_id": group_id,
        "status": "Optimal",
        "cost_usd": round(cost, 6),
        "foods": food_df,
        "nutrients": pd.DataFrame(nut_rows),
    }

In [12]:
def print_results(result: dict) -> None:
    gid    = result["group_id"]
    status = result["status"]
 
    print(f"\nGROUP : {gid.upper().replace('_', ' ')}")

    # Status
    if status != "Optimal":
        print("No feasible solution found.")
        return
    
    print(f"    Minimum lunch cost: ${result['cost_usd']:.4f} USD")

    # Optimal foods
    print(f"\nOptimal lunch ingredients")

    food_df = result["foods"].copy()
    food_df["grams"] = food_df["grams"].map(lambda v: f"{v:.1f} g")
    food_df["cost_usd"] = food_df["cost_usd"].map(lambda v: f"${v:.5f}")
    food_df["cost_pct"] = food_df["cost_pct"].map(lambda v: f"{v:.1f}%")
    food_df.index = food_df.index + 1
 
    print(tabulate(
        food_df,
        headers=["#", "Food", "Portion", "Cost (USD)", "Cost share"],
        tablefmt="rounded_outline",
        showindex=True,
    ))

    # Nutritional achievement
    print(f"\nNutritional Achievement")

    nut_df = result["nutrients"]
    display_rows = []
    for _, row in nut_df.iterrows():
        flag  = ""
        if row["lb_binding"]: flag += "[LB]"
        if row["ub_binding"]: flag += "[UB]"
        display_rows.append([
            row["nutrient"],
            f"{row['achieved']:.4f}",
            f"{row['lb']:.3f}" if not np.isnan(row["lb"]) else "-",
            f"{row['ub']:.3f}" if not np.isnan(row["ub"]) else "-",
            f"{row['lb_slack']:.4f}" if not np.isnan(row["lb_slack"]) else "-",
            f"{row['ub_slack']:.4f}" if not np.isnan(row["ub_slack"]) else "-",
            flag,
        ])
 
    print(tabulate(
        display_rows,
        headers=["Nutrient", "Achieved", "LB", "UB", "LB slack", "UB slack", "Binding"],
        tablefmt="rounded_outline",
    ))

    # Summary
    binding_lb    = nut_df[nut_df["lb_binding"]]["nutrient"].tolist()
    binding_ub    = nut_df[nut_df["ub_binding"]]["nutrient"].tolist()
    n_foods       = len(result["foods"])
    top_cost_food = (
        result["foods"].iloc[0]["food_name"] if n_foods > 0 else "-"
    )
 
    print(f"\nSummary")
    print(f"   Minimum cost      : ${result['cost_usd']:.4f} per lunch")
    print(f"   Foods selected    : {n_foods} / 204 available")
    print(f"   Highest cost item : {top_cost_food}")
    print()
    if binding_lb:
        print("   Binding lower bounds (scarce nutrients driving cost up):")
        for b in binding_lb:
            print(f"    - {b}")
    else:
        print("   No binding lower bounds.")
    if binding_ub:
        print("   Binding upper bounds (limits on cheap food usage):")
        for b in binding_ub:
            print(f"    - {b}")
    else:
        print("   No binding upper bounds.")
    print()

In [13]:
# Solve LP for each age-sex group
all_results = []
 
for group_id in lb.index:
    result = solve_lp(
        group_id        = group_id,
        c               = c,
        A               = A,
        food_names      = food_names,
        nut_cols        = nut_cols,
        lb_row          = lb.loc[group_id],
        ub_row          = ub.loc[group_id],
        constraint_sets = constraint_sets,
        groups          = groups,
    )
    all_results.append(result)
    print_results(result)


GROUP : CHILDREN 4 8 MF
    Minimum lunch cost: $0.7789 USD

Optimal lunch ingredients
╭─────┬────────────────────────────────────────┬───────────┬──────────────┬──────────────╮
│   # │ Food                                   │ Portion   │ Cost (USD)   │ Cost share   │
├─────┼────────────────────────────────────────┼───────────┼──────────────┼──────────────┤
│   1 │ Cornmeal, Whole Grain, Yellow (dry)    │ 73.0 g    │ $0.21458     │ 27.6%        │
│   2 │ Salmon, Pink, Canned (drained)         │ 22.8 g    │ $0.19048     │ 24.5%        │
│   3 │ Milk, 2% Reduced Fat, Fluid            │ 150.0 g   │ $0.14267     │ 18.3%        │
│   4 │ Flaxseeds, Whole                       │ 13.4 g    │ $0.10332     │ 13.3%        │
│   5 │ Collard Greens, Raw                    │ 13.0 g    │ $0.05157     │ 6.6%         │
│   6 │ Mayonnaise, Regular                    │ 10.2 g    │ $0.05153     │ 6.6%         │
│   7 │ Vegetable Oil (Soybean/Canola blend)   │ 3.0 g     │ $0.01057     │ 1.4%         │
│ 

In [14]:
# Print and save the comparison
def save_as_md(result: dict, save_path: str = ".") -> None:
    if result["status"] != "Optimal":
        print(f"  Skipping {result['group_id']} — no optimal solution.")
        return

    gid = result["group_id"]
    filename = f"results_{gid}.md"

    lines = []

    # Header
    lines.append(f"# LP Results — {gid.replace('_', ' ').title()}\n")
    lines.append(f"**Solver status:** {result['status']}  ")
    lines.append(f"**Minimum lunch cost:** ${result['cost_usd']:.4f} USD\n")

    # Optimal food
    lines.append("## Optimal Lunch Foods\n")
    lines.append(
        "The LP selected these foods and gram quantities to satisfy all "
        "nutritional constraints at the lowest possible cost.\n"
    )
    lines.append("| # | Food | Portion (g) | Cost (USD) | Cost share (%) |")
    lines.append("|---|------|------------:|------------:|---------------:|")

    food_df = result["foods"].copy()
    for idx, row in food_df.iterrows():
        lines.append(
            f"| {idx + 1} "
            f"| {row['food_name']} "
            f"| {row['grams']:.1f} "
            f"| ${row['cost_usd']:.5f} "
            f"| {row['cost_pct']:.1f}% |"
        )
    lines.append("")

    # Nutritional Achievement
    lines.append("## Nutritional Achievement vs. DRI Bounds\n")
    lines.append(
        "- **LB slack** = achieved - lower bound (0 = binding constraint)  \n"
        "- **UB slack** = upper bound - achieved (0 = binding constraint)  \n"
        "- **[LB] / [UB]** marks binding constraints — these are the cost drivers.\n"
    )
    lines.append(
        "| Nutrient | Achieved | LB | UB | LB slack | UB slack | Binding |"
    )
    lines.append(
        "|----------|--------:|----:|----:|---------:|---------:|---------|"
    )

    nut_df = result["nutrients"]
    for _, row in nut_df.iterrows():
        flag = ""
        if row["lb_binding"]: flag += "[LB]"
        if row["ub_binding"]: flag += " [UB]"
        lb_s = f"{row['lb']:.3f}"       if not np.isnan(row["lb"])       else "—"
        ub_s = f"{row['ub']:.3f}"       if not np.isnan(row["ub"])       else "—"
        lb_sl = f"{row['lb_slack']:.4f}" if not np.isnan(row["lb_slack"]) else "—"
        ub_sl = f"{row['ub_slack']:.4f}" if not np.isnan(row["ub_slack"]) else "—"
        lines.append(
            f"| {row['nutrient']} "
            f"| {row['achieved']:.4f} "
            f"| {lb_s} "
            f"| {ub_s} "
            f"| {lb_sl} "
            f"| {ub_sl} "
            f"| {flag.strip()} |"
        )
    lines.append("")

    # Summary
    binding_lb = nut_df[nut_df["lb_binding"]]["nutrient"].tolist()
    binding_ub = nut_df[nut_df["ub_binding"]]["nutrient"].tolist()
    n_foods = len(result["foods"])
    top_cost_food = result["foods"].iloc[0]["food_name"] if n_foods > 0 else "—"

    lines.append("## Summary\n")
    lines.append(f"- **Minimum cost:** ${result['cost_usd']:.4f} per lunch")
    lines.append(f"- **Foods selected:** {n_foods} / 204 available")
    lines.append(f"- **Highest cost item:** {top_cost_food}")
    lines.append("")

    if binding_lb:
        lines.append("**Binding lower bounds** (scarce nutrients driving cost up):\n")
        for b in binding_lb:
            lines.append(f"- {b}")
        lines.append("")

    if binding_ub:
        lines.append("**Binding upper bounds** (limits on cheap food usage):\n")
        for b in binding_ub:
            lines.append(f"- {b}")
        lines.append("")

    # Write and save
    os.makedirs(save_path, exist_ok=True)
    filepath = os.path.join(save_path, filename)

    with open(filepath, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"  Saved: {filename}")

In [15]:
def print_comparison(all_results: list[dict]) -> None:
    rows = []
    for r in all_results:
        if r["status"] == "Optimal":
            n_binding = (
                r["nutrients"]["lb_binding"].sum()
                + r["nutrients"]["ub_binding"].sum()
            )
            n_foods = len(r["foods"])
            cost    = f"${r['cost_usd']:.4f}"
        else:
            n_binding, n_foods, cost = "-", "-", "INFEASIBLE"
 
        rows.append([r["group_id"], r["status"], cost, n_foods, int(n_binding)])
 
    print(tabulate(
        rows,
        headers=["Group", "Status", "Min cost (USD)", "Foods used", "Binding"],
        tablefmt="rounded_outline",
    ))
    print()

In [16]:
print_comparison(all_results)

╭─────────────────┬──────────┬──────────────────┬──────────────┬───────────╮
│ Group           │ Status   │ Min cost (USD)   │   Foods used │   Binding │
├─────────────────┼──────────┼──────────────────┼──────────────┼───────────┤
│ children_4_8_MF │ Optimal  │ $0.7789          │           10 │         9 │
│ males_9_13      │ Optimal  │ $0.9489          │           10 │         8 │
│ females_9_13    │ Optimal  │ $0.9416          │            9 │         9 │
│ males_14_18     │ Optimal  │ $1.0850          │           10 │         9 │
│ females_14_18   │ Optimal  │ $1.0610          │            8 │         7 │
╰─────────────────┴──────────┴──────────────────┴──────────────┴───────────╯



In [17]:
import random
import pulp

In [18]:
print(nutrition.columns.tolist())

['food_id', 'food_name', 'category', 'subcategory', 'fdc_id', 'preparation_state', 'energy_kcal', 'protein_g', 'total_fat_g', 'carbohydrates_g', 'dietary_fiber_g', 'sugars_g', 'saturated_fat_g', 'monounsaturated_fat_g', 'polyunsaturated_fat_g', 'cholesterol_mg', 'calcium_mg', 'iron_mg', 'magnesium_mg', 'phosphorus_mg', 'potassium_mg', 'sodium_mg', 'zinc_mg', 'vitamin_a_ug_rae', 'vitamin_d_ug', 'vitamin_e_mg_ate', 'vitamin_k_ug', 'vitamin_c_mg', 'thiamin_b1_mg', 'riboflavin_b2_mg', 'niacin_b3_mg', 'vitamin_b6_mg', 'folate_ug_dfe', 'vitamin_b12_ug', 'selenium_ug']


In [19]:
import pandas as pd
import numpy as np

print(f"Oryginalna liczba produktów w bazie: {len(nutrition)}")

# 1. Definiujemy słowa kluczowe do usunięcia (płatki śniadaniowe, podroby, surowe mąki)
banned_keywords = ['cereal', 'flakes', 'bran', 'cornmeal', 'flaxseeds', 'flour', 'liver', 'heart', 'kidney']
pattern = '|'.join(banned_keywords)

# Kopiujemy bazę i usuwamy zakazane produkty po nazwie
filtered_nutrition = nutrition[~nutrition['food_name'].str.contains(pattern, case=False, na=False)].copy()

# 2. Blokada surowych warzyw liściastych oraz surowych strączków
def is_invalid_raw(row):
    name = str(row['food_name']).lower()
    # Te warzywa i strączki w szkolnym obiedzie nie mogą być podane na surowo (raw)
    invalid_raw_items = ['collard', 'cabbage', 'kale', 'spinach', 'broccoli', 'beans', 'chickpeas', 'lentils', 'peas']
    if 'raw' in name and any(x in name for x in invalid_raw_items):
        return True
    return False

filtered_nutrition['is_raw_bug'] = filtered_nutrition.apply(is_invalid_raw, axis=1)
filtered_nutrition = filtered_nutrition[filtered_nutrition['is_raw_bug'] == False].drop(columns=['is_raw_bug'])

# 3. Magiczna synchronizacja indeksów (Klucz do braku błędów!)
# Pobieramy numery wierszy, które przetrwały nasze czyszczenie
valid_indices = filtered_nutrition.index.tolist()

# Nadpisujemy globalną listę nazw potraw dla solvera
food_names = filtered_nutrition['food_name'].tolist()

# Docinamy wektor cen 'c' do ocalałych produktów
if isinstance(c, np.ndarray):
    c = c[valid_indices]
else:
    c = [c[i] for i in valid_indices]

# Docinamy macierz wartości odżywczych 'A' do ocalałych produktów
if isinstance(A, np.ndarray):
    A = A[valid_indices, :]
else:
    A = [A[i] for i in valid_indices]

# Aktualizujemy zmienną 'n' (łączna liczba produktów, którą czyta Twój solver)
n = len(filtered_nutrition)

print(f"Sukces! Nowa liczba produktów po kulinarnym oczyszczeniu: {n}")

Oryginalna liczba produktów w bazie: 204
Sukces! Nowa liczba produktów po kulinarnym oczyszczeniu: 192


In [ ]:
import pulp
import random
import pandas as pd

def solve_lp_weekly_shared(group_name_base, c, A, food_names, nut_cols, 
                           lb_male, ub_male, lb_female, ub_female, constraint_sets):
    """
    Optimizes a single, shared weekly menu for males and females of the same age group.
    The selected food items are identical each day, but the portions (grammages) can vary.
    """
    n = len(food_names)
    days = [1, 2, 3, 4, 5]
    nut_idx = {nut: j for j, nut in enumerate(nut_cols)}
    
    # Create the minimization problem for total combined cost (Male + Female)
    prob = pulp.LpProblem(f"Shared_Menu_{group_name_base}", pulp.LpMinimize)

    # --- 1. VARIABLES ---
    # Grammages (separate for males and females) - lowered upBound from 400 to 250 for realism
    x_m = pulp.LpVariable.dicts("x_m", ((i, t) for i in range(n) for t in days), lowBound=0, upBound=250.0)
    x_f = pulp.LpVariable.dicts("x_f", ((i, t) for i in range(n) for t in days), lowBound=0, upBound=250.0)
    
    # SHARED binary variable: determines if a food item enters the daily menu at all
    z_shared = pulp.LpVariable.dicts("z_shared", ((i, t) for i in range(n) for t in days), cat='Binary')
    
    # Shared main dish category rotation
    main_subgroups = ['Poultry', 'Beef', 'Pork', 'Lamb', 'Fish & Seafood', 'Eggs', 'Legumes']
    y_shared = pulp.LpVariable.dicts("y_shared", ((cat, t) for cat in main_subgroups for t in days), cat='Binary')
    
    # --- 2. BEVERAGES AND SOLID FOOD SEPARATION ---
    def get_indices(cats):
        return [idx for idx, name in enumerate(food_names) 
                if nutrition.iloc[idx]['category'] in cats]

    milk_indices, _ = groups.get("fluid_milk", ([], 250.0))
    base_drink_cats = get_indices(['Beverages', 'Water', 'Juices', 'Milk'])
    drink_indices = list(set(base_drink_cats + milk_indices))
    food_indices = [i for i in range(n) if i not in drink_indices]

    # --- 3. OBJECTIVE FUNCTION (Minimize total cost of both portions) ---
    obj_terms = []
    for i in range(n):
        for t in days:
            random_offset = random.uniform(0, 0.0001)
            # Combined cost = (price * male_grams) + (price * female_grams)
            cost_m = (c[i] + random_offset) * x_m[i, t]
            cost_f = (c[i] + random_offset) * x_f[i, t]
            # Diversity penalty (applied once since the selected menu is shared)
            diversity_penalty = z_shared[i, t] * 0.02 
            obj_terms.append(cost_m + cost_f + diversity_penalty)
            
    prob += pulp.lpSum(obj_terms)


    # --- 4. DAILY CONSTRAINTS (For each day t) ---
    for t in days:
        food_mass_m = pulp.lpSum([x_m[i, t] for i in food_indices])
        food_mass_f = pulp.lpSum([x_f[i, t] for i in food_indices])
        
        # --- SHARED MENU CONSTRAINTS ---
        if drink_indices:
            # Both individuals receive a 250g drink portion
            prob += pulp.lpSum([x_m[i, t] for i in drink_indices]) == 250.0
            prob += pulp.lpSum([x_f[i, t] for i in drink_indices]) == 250.0
            # Maximum of 1 shared beverage item per day
            prob += pulp.lpSum([z_shared[i, t] for i in drink_indices]) <= 1
            
        if milk_indices:
            prob += pulp.lpSum([z_shared[i, t] for i in milk_indices]) <= 1

        # --- PORTION LOGIC (Binding grammages to the single shared binary variable) ---
        for i in range(n):
            if i in drink_indices:
                # If a beverage is selected (z_shared=1), both get 250g. If z_shared=0, both get 0g.
                prob += x_m[i, t] == 250.0 * z_shared[i, t]
                prob += x_f[i, t] == 250.0 * z_shared[i, t]
            else:
                # [POPRAWKA]: Inteligentne limity dolne. Zapobiegają porcjom po 20g ryżu czy warzyw.
                # Dla tłuszczów/sosów dopuszczamy małe porcje (od 10g), dla reszty twarde minimum 50g.
                is_fat = any(x in food_names[i].lower() for x in ['oil', 'mayonnaise', 'margarine', 'butter', 'dressing'])
                min_grams = 10.0 if is_fat else 50.0 
                
                prob += x_m[i, t] >= min_grams * z_shared[i, t]
                prob += x_m[i, t] <= 250.0 * z_shared[i, t]
                
                prob += x_f[i, t] >= min_grams * z_shared[i, t]
                prob += x_f[i, t] <= 250.0 * z_shared[i, t]

        # --- EXACTLY ONE MAIN DISH PER DAY ---
        daily_main_vars = []
        for cat in main_subgroups:
            c_idx = [idx for idx in get_indices([cat]) if idx in food_indices]
            if c_idx:
                # Bind the main item to the shared category tracker y_shared
                prob += pulp.lpSum([x_m[i, t] for i in c_idx]) <= 250 * y_shared[cat, t]
                prob += pulp.lpSum([x_f[i, t] for i in c_idx]) <= 250 * y_shared[cat, t]
                
                # [POPRAWKA]: Podniesienie dolnej granicy całego dania głównego z 50g do 100g, 
                # aby główny element posiłku był odpowiednio sycący.
                prob += pulp.lpSum([x_m[i, t] for i in c_idx]) >= 100 * y_shared[cat, t]
                prob += pulp.lpSum([x_f[i, t] for i in c_idx]) >= 100 * y_shared[cat, t]
                daily_main_vars.append(y_shared[cat, t])
        
        if daily_main_vars:
            prob += pulp.lpSum(daily_main_vars) == 1

        # --- FRUIT AND VEGETABLE SHARE (MyPlate ~40% independently for each gender) ---
        vf_indices = [idx for idx in get_indices(['Vegetables', 'Fruits']) if idx in food_indices]
        if vf_indices:
            prob += pulp.lpSum([x_m[i, t] for i in vf_indices]) >= 0.4 * food_mass_m
            prob += pulp.lpSum([x_f[i, t] for i in vf_indices]) >= 0.4 * food_mass_f

        # --- INDEPENDENT NUTRITIONAL TARGETS FOR MALES AND FEMALES ---
        for nut in constraint_sets["min_only"] + constraint_sets["range"]:
            j = nut_idx[nut]
            # Male minimum targets
            val_m = lb_male.get(nut, "")
            if val_m != "" and not pd.isna(val_m):
                prob += pulp.lpSum([A[i, j] * x_m[i, t] for i in range(n)]) >= float(val_m)
            # Female minimum targets
            val_f = lb_female.get(nut, "")
            if val_f != "" and not pd.isna(val_f):
                prob += pulp.lpSum([A[i, j] * x_f[i, t] for i in range(n)]) >= float(val_f)
        
        for nut in constraint_sets["max_only"] + constraint_sets["range"]:
            if nut == "sodium_mg": continue 
            j = nut_idx[nut]
            # Male maximum limits
            val_m = ub_male.get(nut, "")
            if val_m != "" and not pd.isna(val_m):
                prob += pulp.lpSum([A[i, j] * x_m[i, t] for i in range(n)]) <= float(val_m)
            # Female maximum limits
            val_f = ub_female.get(nut, "")
            if val_f != "" and not pd.isna(val_f):
                prob += pulp.lpSum([A[i, j] * x_f[i, t] for i in range(n)]) <= float(val_f)

    # --- 5. WEEKLY CONSTRAINTS (Separate sodium, shared variety rotation) ---
    s_idx = nut_idx["sodium_mg"]
    prob += pulp.lpSum([A[i, s_idx] * x_m[i, t] for i in range(n) for t in days]) <= 1500 * 5
    prob += pulp.lpSum([A[i, s_idx] * x_f[i, t] for i in range(n) for t in days]) <= 1500 * 5

    # Main category rotation (shared school-wide restriction)
    for cat in main_subgroups:
        c_idx = [idx for idx in get_indices([cat]) if idx in food_indices]
        if c_idx:
            prob += pulp.lpSum([y_shared[cat, t] for t in days]) <= 2

    # Item repetition restriction across the week
    for i in range(n):
        if i in drink_indices:
            prob += pulp.lpSum([z_shared[i, t] for i in days]) <= 5
        else:
            prob += pulp.lpSum([z_shared[i, t] for i in days]) <= 2

    # --- 6. SOLVER ---
    status = prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=90)) 

    if pulp.LpStatus[status] != 'Optimal':
        return {"status": "Infeasible", "group_id": group_name_base, "days": []}

    # --- 7. RESULTS ---
    weekly_results = []
    for t in days:
        day_foods = []
        for i in range(n):
            val_m = pulp.value(x_m[i, t])
            val_f = pulp.value(x_f[i, t])
            
            # If the food item was included for at least one of the genders
            if (val_m and val_m > 0.1) or (val_f and val_f > 0.1):
                day_foods.append({
                    "food": food_names[i], 
                    "grams_male": round(val_m or 0.0, 2),
                    "grams_female": round(val_f or 0.0, 2),
                    "cost_male": round((val_m or 0.0) * c[i], 4),
                    "cost_female": round((val_f or 0.0) * c[i], 4)
                })
        weekly_results.append(day_foods)

    return {
        "group_id": group_name_base, 
        "status": "Optimal",
        "total_cost_usd": round(pulp.value(prob.objective), 4), 
        "days": weekly_results
    }

In [27]:
from tabulate import tabulate

# --- 1. CONFIGURATION: AGE GROUP PAIRING ---
# Define the school menu structures. Children 4-8 are planned together (same targets),
# while older groups are paired up: (base_id, male_id, female_id)
shared_groups_config = [
    ("ages_4_8_shared", "children_4_8_MF", "children_4_8_MF"), # Youngest group shares the same nutritional targets
    ("ages_9_13_shared", "males_9_13", "females_9_13"),
    ("ages_14_18_shared", "males_14_18", "females_14_18")
]

weekly_menu_results = []

# --- 2. EXECUTE OPTIMIZATION ---
for target_name, male_id, female_id in shared_groups_config:
    print(f"Calculating shared weekly menu for: {target_name} ({male_id} + {female_id})...")
    
    # Extract rows from your lb and ub DataFrames for both genders
    lb_male = lb.loc[male_id]
    ub_male = ub.loc[male_id]
    lb_female = lb.loc[female_id]
    ub_female = ub.loc[female_id]
    
    # Call the new shared LP optimization function
    res = solve_lp_weekly_shared(
        target_name, c, A, food_names, nut_cols, 
        lb_male, ub_male, lb_female, ub_female, constraint_sets
    )
    weekly_menu_results.append(res)


# --- 3. DISPLAY RESULTS ---
for result in weekly_menu_results:
    print("\n" + "="*85)
    print(f" SHARED WEEKLY SCHOOL MENU (USA): {result['group_id'].upper()} ")
    print("="*85)
    
    if result["status"] != "Optimal":
        print(f"Status: {result['status']} - Could not find a feasible shared 5-day menu.")
        continue
        
    print(f"Total Combined Weekly Cost (1 Male + 1 Female): ${result['total_cost_usd']:.2f}")
    
    for day_idx, day_content in enumerate(result["days"], 1):
        print(f"\n--- DAY {day_idx} ---")
        
        table_data = []
        for f in day_content:
            # Format portion sizes - if the 4-8 group has identical values,
            # it will neatly display the same amount in both columns.
            portion_m = f"{f['grams_male']}g" if f['grams_male'] > 0 else "-"
            portion_f = f"{f['grams_female']}g" if f['grams_female'] > 0 else "-"
            
            # Combine item costs for both portions together
            total_item_cost = f['cost_male'] + f['cost_female']
            
            table_data.append([
                f["food"], 
                portion_m, 
                portion_f, 
                f"${total_item_cost:.4f}"
            ])
            
        # Display a clean, structured table with separate columns for male and female portions
        headers = ["Food Item", "Portion Male", "Portion Female", "Combined Cost"]
        print(tabulate(table_data, headers=headers, tablefmt="rounded_outline"))

print("\nOptimization Finished.")

Calculating shared weekly menu for: ages_4_8_shared (children_4_8_MF + children_4_8_MF)...
Calculating shared weekly menu for: ages_9_13_shared (males_9_13 + females_9_13)...
Calculating shared weekly menu for: ages_14_18_shared (males_14_18 + females_14_18)...

 SHARED WEEKLY SCHOOL MENU (USA): AGES_4_8_SHARED 
Total Combined Weekly Cost (1 Male + 1 Female): $20.19

--- DAY 1 ---
╭───────────────────────────────────────┬────────────────┬──────────────────┬─────────────────╮
│ Food Item                             │ Portion Male   │ Portion Female   │ Combined Cost   │
├───────────────────────────────────────┼────────────────┼──────────────────┼─────────────────┤
│ Tuna, Canned in Water (drained)       │ 105.64g        │ 105.64g          │ $1.9342         │
│ Milk, Almond, Unsweetened (fortified) │ 250.0g         │ 250.0g           │ $0.9244         │
│ Barley, Pearl (cooked)                │ 50.0g          │ 50.0g            │ $0.3304         │
│ Broccoli, Frozen (boiled)             

## References

### Nutritional Data

U.S. Department of Agriculture, Agricultural Research Service. (2019).
*FoodData Central* (SR Legacy Foods / Foundation Foods).
Retrieved from https://fdc.nal.usda.gov

### Food Price Data

U.S. Bureau of Labor Statistics. (2024).
*Average retail food prices — CPI average price series*.
U.S. Department of Labor.
Retrieved from https://data.bls.gov

U.S. Department of Agriculture, Economic Research Service. (2024).
*Meat price spreads*.
Retrieved from https://www.ers.usda.gov/data-products/meat-price-spreads

U.S. Department of Agriculture, Economic Research Service. (2024).
*Fruit and vegetable prices*.
Retrieved from https://www.ers.usda.gov/topics/food-markets-prices

### Dietary Reference Intakes (DRI)

Institute of Medicine, Food and Nutrition Board. (1997).
*Dietary reference intakes for calcium, phosphorus, magnesium, vitamin D,
and fluoride*. National Academies Press.
https://doi.org/10.17226/5776

Institute of Medicine, Food and Nutrition Board. (1998).
*Dietary reference intakes for thiamin, riboflavin, niacin, vitamin B6,
folate, vitamin B12, pantothenic acid, biotin, and choline*.
National Academies Press.
https://doi.org/10.17226/6015

Institute of Medicine, Food and Nutrition Board. (2000).
*Dietary reference intakes for vitamin C, vitamin E, selenium,
and carotenoids*. National Academies Press.
https://doi.org/10.17226/9810

Institute of Medicine, Food and Nutrition Board. (2001).
*Dietary reference intakes for vitamin A, vitamin K, arsenic, boron,
chromium, copper, iodine, iron, manganese, molybdenum, nickel, silicon,
vanadium, and zinc*. National Academies Press.
https://doi.org/10.17226/10026

Institute of Medicine, Food and Nutrition Board. (2005).
*Dietary reference intakes for energy, carbohydrate, fiber, fat, fatty
acids, cholesterol, protein, and amin